In [1]:
!git clone https://github.com/unza-speech-lab/BembaSpeech.git


Cloning into 'BembaSpeech'...
remote: Enumerating objects: 35745, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 35745 (delta 169), reused 327 (delta 141), pack-reused 35383 (from 1)
Receiving objects: 100% (35745/35745), 2.44 GiB | 19.36 MiB/s, done.
Resolving deltas: 100% (17741/17741), done.
Updating files: 100% (14444/14444), done.


In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets evaluate accelerate
!pip install scikit-learn pandas numpy
!pip install jiwer  # WER metric backend
!pip install huggingface_hub


Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.1 MB/s eta 0:00:00


In [3]:
# -*- coding: utf-8 -*-
"""ASR_BEMBA_WHISPER.ipynb"""

import os
import re
import gc
import math
import shutil
import torch
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List

from datasets import Dataset, Audio, concatenate_datasets
from evaluate import load
from sklearn.model_selection import train_test_split
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
import transformers

print("Transformers version:", transformers.__version__)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
set_seed(42)

# ============================================================
# 2  Paths — dataset, audio, model and output directories
# ============================================================

DATA_DIR   = "/content/BembaSpeech/data/"
AUDIO_DIR  = os.path.join(DATA_DIR, "audio/")
TRAIN_TSV  = os.path.join("/content/BembaSpeech/data/splits/train.csv")
OUTPUT_DIR = "./bemba-whisper-model"
FINAL_DIR  = "./bemba-whisper-final"
MODEL_NAME = "buumba641/bemba-whisper-medium-shona-checkpoint"

PROCESSED_DATA_DIR  = "./processed_data"
PROCESSED_TRAIN_DIR = os.path.join(PROCESSED_DATA_DIR, "train")
PROCESSED_VAL_DIR   = os.path.join(PROCESSED_DATA_DIR, "val")

# ============================================================
# 3  Load Bemba dataset from CSV/TSV and verify audio files exist
# ============================================================

print("Loading Bemba dataset...")

df = pd.read_csv(TRAIN_TSV)
df = df[["audio", "sentence"]].rename(columns={"sentence": "text"})
df["path"] = AUDIO_DIR + df["audio"]
df = df.dropna()

total_before = len(df)
df["exists"]  = df["path"].apply(os.path.exists)
found_df      = df[df["exists"]]
missing_df    = df[~df["exists"]]

print(f"Total rows in TSV (after dropna): {total_before}")
print(f"Audio files FOUND:   {len(found_df)}")
print(f"Audio files MISSING: {len(missing_df)}")

if len(missing_df) > 0:
    print(f"Missing rate: {len(missing_df)/total_before*100:.1f}%")
    for _, row in missing_df.head(10).iterrows():
        print(f"  {row['path']}")

df = df[df["exists"]][["path", "text"]].reset_index(drop=True)
print(f"\nUsable samples: {len(df)}")

# ============================================================
# 4  Train / validation split (90% / 10%)
# ============================================================

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

# ============================================================
# 5  Convert to HuggingFace Dataset and cast audio to 16 kHz
# ============================================================

def create_dataset(df):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
    return dataset

train_dataset = create_dataset(train_df)
val_dataset   = create_dataset(val_df)

# ============================================================
# 6  Clean text — lowercase, strip punctuation, digits and non-ASCII (Bemba)
# ============================================================

CHARS_TO_REMOVE    = r'[,?.!\-;:""%\'"\'\u2018\u2019\u2026()[\]{}]'
EXTRA_LETTERS_TO_STRIP = r'[qx]'

def clean_text(batch):
    text = batch["text"].lower()
    text = re.sub(CHARS_TO_REMOVE, "", text)
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(EXTRA_LETTERS_TO_STRIP, "", text)
    text = re.sub(r"[^\x00-\x7F]", "", text)
    text = re.sub(r" +", " ", text).strip()
    batch["text"] = text
    return batch

train_dataset = train_dataset.map(clean_text)
val_dataset   = val_dataset.map(clean_text)

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("\n--- Sample cleaned text ---")
for i in range(5):
    print(repr(train_dataset[i]["text"]))
print("---\n")



Transformers version: 5.12.1
Loading Bemba dataset...
Total rows in TSV (after dropna): 11906
Audio files FOUND:   11906
Audio files MISSING: 0

Usable samples: 11906
Train: 10715
Validation: 1191


Map:   0%|          | 0/10715 [00:00<?, ? examples/s]

Map:   0%|          | 0/1191 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10715 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1191 [00:00<?, ? examples/s]


--- Sample cleaned text ---
'joni wena afwele icina america nga filyafine afwala na kale'
'nashala mumenso poko'
'bushe ifisangwa mu ciputulwa citila mukati ka christian brethren filafwilisha'
'cembe pa kumfwa amashiwi ya musango uyu camupesha amano'
'cishima nshi ico mulekabila ukwikalapo'
---



In [4]:
import datasets
datasets.disable_caching()


In [5]:
# ============================================================
# 7  Whisper processor — force Swahili as phonetic proxy language for Bemba
# ============================================================
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

# language="swahili" as a phonetic proxy for Bemba
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="swahili", task="transcribe")
processor  = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# ============================================================
# 8  Prepare dataset — extract features in chunks, cache to disk to avoid OOM
# ============================================================
NUM_PROC   = min(4, os.cpu_count() or 1)
CHUNK_SIZE = 500

TRAIN_CHUNKS_DIR = os.path.join(PROCESSED_DATA_DIR, "train_chunks")
VAL_CHUNKS_DIR   = os.path.join(PROCESSED_DATA_DIR, "val_chunks")

def prepare_dataset(batch):
    audio = batch["path"]
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

def process_in_chunks(dataset, chunks_dir, chunk_size=CHUNK_SIZE, num_proc=NUM_PROC):
    """Feature-extract `dataset` in disposable-process chunks, resumable per chunk."""
    os.makedirs(chunks_dir, exist_ok=True)
    n_chunks = math.ceil(len(dataset) / chunk_size)
    chunk_dirs = []

    for i in range(n_chunks):
        chunk_dir = os.path.join(chunks_dir, f"chunk_{i:04d}")
        chunk_dirs.append(chunk_dir)

        if os.path.isdir(chunk_dir):
            try:
                Dataset.load_from_disk(chunk_dir)
                print(f"  chunk {i+1}/{n_chunks}: already processed, skipping")
                continue
            except Exception:
                print(f"  chunk {i+1}/{n_chunks}: found but looks incomplete, redoing")
                shutil.rmtree(chunk_dir)

        start, end = i * chunk_size, min((i + 1) * chunk_size, len(dataset))
        shard = dataset.select(range(start, end))
        shard_num_proc = max(1, min(num_proc, len(shard)))

        print(f"  chunk {i+1}/{n_chunks}: rows {start}:{end}, num_proc={shard_num_proc}")
        shard = shard.map(
            prepare_dataset,
            remove_columns=shard.column_names,
            num_proc=shard_num_proc,
            keep_in_memory=False,
            writer_batch_size=50,
        )
        shard.save_to_disk(chunk_dir)

        del shard
        gc.collect()

    chunks = [Dataset.load_from_disk(d) for d in chunk_dirs]
    return concatenate_datasets(chunks) if len(chunks) > 1 else chunks[0]

processed_data_exists = os.path.isdir(PROCESSED_TRAIN_DIR) and os.path.isdir(PROCESSED_VAL_DIR)

if processed_data_exists:
    try:
        print(f"Found cached processed dataset in '{PROCESSED_DATA_DIR}', loading from disk...")
        train_dataset = Dataset.load_from_disk(PROCESSED_TRAIN_DIR)
        val_dataset   = Dataset.load_from_disk(PROCESSED_VAL_DIR)
    except Exception as e:
        print(f"Cached dataset looked incomplete ({e}); recomputing instead.")
        processed_data_exists = False

if not processed_data_exists:
    print("Processing train split...")
    train_dataset = process_in_chunks(train_dataset, TRAIN_CHUNKS_DIR)
    print("Processing validation split...")
    val_dataset = process_in_chunks(val_dataset, VAL_CHUNKS_DIR)

    print(f"Saving processed datasets to disk at '{PROCESSED_DATA_DIR}'...")
    train_dataset.save_to_disk(PROCESSED_TRAIN_DIR)
    val_dataset.save_to_disk(PROCESSED_VAL_DIR)

    del train_dataset, val_dataset
    gc.collect()

    train_dataset = Dataset.load_from_disk(PROCESSED_TRAIN_DIR)
    val_dataset   = Dataset.load_from_disk(PROCESSED_VAL_DIR)

print("Train after preparation:", len(train_dataset))
print("Validation after preparation:", len(val_dataset))

# ============================================================
# 9  Data collator — pad features and labels into batches
# ============================================================

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

Processing train split...
  chunk 1/22: rows 0:500, num_proc=2


Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

...

Saving processed datasets to disk at './processed_data'...


Saving the dataset (0/21 shards):   0%|          | 0/10715 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1191 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

Train after preparation: 10715
Validation after preparation: 1191


In [6]:
# ============================================================
# 10  Load model and set forced decoder language / task
# ============================================================
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="swahili", task="transcribe")
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.suppress_tokens    = []
model.generation_config.language           = "swahili"
model.generation_config.task               = "transcribe"

print(f"Model: {MODEL_NAME}  |  Parameters: {model.num_parameters():,}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=500,
    save_steps=500,
    save_strategy="steps",
    num_train_epochs=1.5,
    learning_rate=1e-5,
    warmup_steps=300,
    weight_decay=0.01,
    gradient_checkpointing=True,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)

# ============================================================
# 11  Clear legacy suppress_tokens on model config
# ============================================================
model.config.suppress_tokens = None

# ============================================================
# 12  Word Error Rate (WER) metric for evaluation
# ============================================================
wer_metric = load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ============================================================
# 13  Build Seq2SeqTrainer
# ============================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

# ============================================================
# 14  Train the model
# ============================================================
print("\n========== STARTING TRAINING ==========")
print(f"Model                : {MODEL_NAME}")
print(f"Transformers version : {transformers.__version__}")
print(f"Train samples        : {len(train_dataset)}")
print(f"Val samples          : {len(val_dataset)}")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate        : {training_args.learning_rate}")
print("========================================\n")

trainer.train()

# ============================================================
# 15  Save final model and processor
# ============================================================
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model and processor saved to {FINAL_DIR}")

# ============================================================
# 16  Final evaluation on validation set
# ============================================================
print("\n========== FINAL EVALUATION ==========")
eval_results = trainer.evaluate()
print(f"Final WER: {eval_results['eval_wer']:.4f}  ({eval_results['eval_wer']*100:.2f}%)")

# ============================================================
# 17  Spot-check inference on 5 validation samples
# ============================================================
def transcribe(idx):
    example = val_dataset[idx]
    device  = model.device

    input_features = torch.tensor(
        example["input_features"]
    ).unsqueeze(0).to(device)

    infer_forced_ids = processor.get_decoder_prompt_ids(language="swahili", task="transcribe")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=infer_forced_ids,
        )

    prediction = processor.tokenizer.decode(
        predicted_ids[0], skip_special_tokens=True
    )

    label_ids = [i for i in example["labels"] if i != -100]
    reference = processor.tokenizer.decode(
        label_ids, skip_special_tokens=True
    )

    print(f"\n--- Sample {idx} ---")
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)

for i in range(5):
    transcribe(i)

print("\n========== TRAINING PIPELINE COMPLETE ==========")


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

Model: buumba641/bemba-whisper-medium-shona-checkpoint  |  Parameters: 763,857,920


...

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor saved to ./bemba-whisper-final

========== FINAL EVALUATION ==========


Training Loss,Validation Loss,Step,Wer
0.061615,0.470381,2010,0.324119


Final WER: 0.3241  (32.41%)

--- Sample 0 ---
REFERENCE : joni ati kanshi shalikapo abene nganda tuleya
PREDICTION: joni ati kanshi alikapo abene ba nganda tuleya

--- Sample 1 ---
REFERENCE : ukwabula ukusambilila mu mitundu imbi
PREDICTION: ukwabula kusambilila mu mitundu imbi

--- Sample 2 ---
REFERENCE : imfumu yafulwa nganshi yatuma abakuyaipaya mulenga
PREDICTION: imfumu yafulwa nganshi yatuma abakuya ipaya mulenga

--- Sample 3 ---
REFERENCE : mu kusondwelela ukwendauka na mafuta ukulasuba bantu amayanda imyotoka ifyakulya ne fintu fimbi tacilefwaikwa kabili cilelanga ukufuma mwisambililo lya mu cipingo cipya
PREDICTION: mu kusondolwela ukwenda uukana amafuta ukulasuba abantu amayanda ni yutoka fyakulya ne fintu fimbi tacilefwaikwa kabili cilelanga ukufuma mu isambililo lya mu cipingo cipya

--- Sample 4 ---
REFERENCE : kuti bamfubu ukwisabuka ku bukali bwa mulilo bamona fye bali mu wingi
PREDICTION: kuti bamukubo ukwisabuka ku bukali bwa mulilo bamona fye bali mu mwingi

======

In [7]:
import os
from huggingface_hub import HfApi, login

# 1. Authenticate with the provided token
HF_TOKEN = os.environ.get("HF_TOKEN")  # set via: export HF_TOKEN=hf_xxx
login(token=HF_TOKEN)

# 2. Define the repository name
repo_id = "bemba-whisper-medium-siwahili-final"

print(f"Pushing model from {FINAL_DIR} to Hugging Face Hub: {repo_id}...")

# 3. Push the model and processor
model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)

print(f"Successfully pushed model to https://huggingface.co/{repo_id}")


Pushing model from ./bemba-whisper-final to Hugging Face Hub: bemba-whisper-medium-siwahili-final...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a4n_bnp/model.safetensors:   0%|          | 99.6kB / 3.06GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Successfully pushed model to https://huggingface.co/bemba-whisper-medium-siwahili-final
